<a href="https://colab.research.google.com/github/motrieu/TC2037-Evidence-3/blob/main/Evidence_3_Matmul_parallel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Small Example to test correctness with N=16. The result should be 2x+1

In [104]:
%%writefile mat_calc.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>

#define N 16
#define THREADS_PER_BLOCK 4

typedef float (*func_ptr)(float);

__global__ void matCalc(float *a, float *b, float *x, float *c, int vec_size) {
  int idx = threadIdx.x + blockIdx.x * blockDim.x;

  int i = idx;
  while (i < vec_size) {
    float sum = 0;
    for (int j= 0; j<vec_size; j++) {
      sum += a[i * vec_size + j] * x[j];
    }

    c[i] = sum + b[i];
    i += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i = 0; i<N; i++) {
    for(int j= 0; j<N; j++) {
      if (i==j) {
        a[j+i*N] = 2;
      } else {
        a[j+i*N] = 0;
      }
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N;i++) {
    b[i] = 1;
  }
}

void fill_x(float *x){
  for(int i=0;i<N;i++) {
    x[i] = i;
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*N*sizeof(float);
  const int vec_size = N*sizeof(float);

  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, vec_size);
	cudaMalloc((void**)&d_c, vec_size);
  cudaMalloc((void**)&d_x, vec_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(vec_size);
	c = (float*)malloc(vec_size);
  x = (float*)malloc(vec_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, vec_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, vec_size, cudaMemcpyHostToDevice);

  matCalc << <N / THREADS_PER_BLOCK, THREADS_PER_BLOCK >> >(d_a, d_b, d_x, d_c, N);

  cudaMemcpy(c, d_c, vec_size, cudaMemcpyDeviceToHost);

  for(int i=0; i<N; i++) {
    std::cout << x[i] << " ";
  }

  std::cout << std::endl;

  for(int i=0; i<N; i++) {
    std::cout << c[i] << " ";
  }
  std::cout << std::endl;

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(a);
  cudaFree(b);
  cudaFree(x);
  cudaFree(c);

  return 0;

}



Overwriting mat_calc.cu


In [105]:
!nvcc -arch=sm_75 mat_calc.cu -o mat_calc
!./mat_calc

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
1 3 5 7 9 11 13 15 17 19 21 23 25 27 29 31 


Timed example with N=16,384

In [126]:
%%writefile mat_calc_timed.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <chrono>
#include <iostream>

#define N 16384
#define THREADS_PER_BLOCK 512

typedef float (*func_ptr)(float);

__global__ void matCalc(float *a, float *b, float *x, float *c, int vec_size) {
  int idx = threadIdx.x + blockIdx.x * blockDim.x;

  int i = idx;
  while (i < vec_size) {
    float sum = 0;
    for (int j= 0; j<vec_size; j++) {
      sum += a[i * vec_size + j] * x[j];
    }

    c[i] = sum + b[i];
    i += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i = 0; i<N; i++) {
    for(int j= 0; j<N; j++) {
      if (i==j) {
        a[j+i*N] = 2;
      } else {
        a[j+i*N] = 0;
      }
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N;i++) {
    b[i] = 1;
  }
}

void fill_x(float *x){
  for(int i=0;i<N;i++) {
    x[i] = i;
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*N*sizeof(float);
  const int vec_size = N*sizeof(float);

  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, vec_size);
	cudaMalloc((void**)&d_c, vec_size);
  cudaMalloc((void**)&d_x, vec_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(vec_size);
	c = (float*)malloc(vec_size);
  x = (float*)malloc(vec_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, vec_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, vec_size, cudaMemcpyHostToDevice);

  // Single Core
  auto start_single = std::chrono::high_resolution_clock::now();
  matCalc << <1, 1 >> >(d_a, d_b, d_x, d_c, N);
  auto stop_single = std::chrono::high_resolution_clock::now();

  // Multi Core
  auto start_multi = std::chrono::high_resolution_clock::now();
  matCalc << <N / THREADS_PER_BLOCK, THREADS_PER_BLOCK >> >(d_a, d_b, d_x, d_c, N);
  auto stop_multi = std::chrono::high_resolution_clock::now();

  cudaMemcpy(c, d_c, vec_size, cudaMemcpyDeviceToHost);

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(a);
  cudaFree(b);
  cudaFree(x);
  cudaFree(c);

  std::cout << "Single Core: " <<  std::chrono::duration_cast< std::chrono::microseconds>(stop_single - start_single).count() << std::endl;
  std::cout << "Multi Core: " << std::chrono::duration_cast< std::chrono::microseconds>(stop_multi - start_multi).count() << std::endl;
  //std::cout << "Speedup: " << std::chrono::duration_cast< std::chrono::microseconds>((stop_single - start_single) / (stop_multi - start_multi)).count() << std::endl;

  return 0;

}



Overwriting mat_calc_timed.cu


In [127]:
!nvcc -arch=sm_75 mat_calc_timed.cu -o mat_calc_timed
!./mat_calc_timed

Single Core: 158
Multi Core: 158
